# Current Portfolio Report Preview

This notebook is the working portfolio-monitor checkpoint for the new universe-first stack. It does three things:

1. Sync the generated `configs/portfolio_monitor/security_reference.csv` from `configs/security_universe.csv`.
2. Build a current position report either from live TWS or by rebuilding from an existing live-report CSV.
3. Show the report pieces we can already generate today: top positions, asset-class exposure, EQ country, US sector, and FI tenor breakdowns.

Recommended kernel: `py313`.


In [1]:
from pathlib import Path
import sys
from typing import Dict

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

def load_local_account_defaults(root: Path) -> Dict[str, str]:
    candidates = [
        root / "configs" / "portfolio_monitor" / "report_accounts.local.env",
        root / "configs" / "report_accounts.local.env",
    ]
    values: Dict[str, str] = {}
    for path in candidates:
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip(chr(34)).strip("'")
        break
    return values

def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

local_account_defaults = load_local_account_defaults(project_root)
artifact_report_dir = project_root / "data" / "artifacts" / "portfolio_monitor"

# Recommended default for reliable notebook replay: rebuild from an existing live CSV.
# Switch SOURCE_MODE to "live" when TWS / IB Gateway is running locally.
SOURCE_MODE = "rebuild_from_existing_csv"  # "live" or "rebuild_from_existing_csv"
EXISTING_LIVE_CSV = first_existing_path([
    project_root / "outputs" / "reports" / "live_ibkr_position_report.csv",
    artifact_report_dir / "live_ibkr_position_report.csv",
])
TARGET_WORKBOOK_PATH = first_existing_path([
    project_root / "outputs" / "reports" / "target_report.xlsx",
    artifact_report_dir / "target_report.xlsx",
])
POSITION_REPORT_OUTPUT = project_root / "outputs" / "reports" / "current_portfolio_report_preview.csv"
RISK_HTML_OUTPUT = project_root / "outputs" / "reports" / "current_portfolio_risk_preview.html"
SECURITY_REFERENCE_OUTPUT = project_root / "configs" / "portfolio_monitor" / "security_reference.csv"

TRY_BUILD_RISK_HTML = False
RETURNS_JSON = None
PROXY_JSON = None
REGIME_JSON = None

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 123
IB_ACCOUNT = local_account_defaults.get("DEFAULT_PROD_ACCOUNT_ID", "")

print(f"Project root: {project_root}")
print(f"Source mode: {SOURCE_MODE}")
print(f"Existing live CSV: {EXISTING_LIVE_CSV}")
print(f"Target workbook: {TARGET_WORKBOOK_PATH}")
print(f"Position report output: {POSITION_REPORT_OUTPUT}")
print(f"Generated security reference: {SECURITY_REFERENCE_OUTPUT}")
print(f"Default live account: {IB_ACCOUNT or '<unset>'}")


Project root: /Users/kelvin/git_projects/market_helper
Source mode: rebuild_from_existing_csv
Existing live CSV: /Users/kelvin/git_projects/market_helper/data/artifacts/portfolio_monitor/live_ibkr_position_report.csv
Target workbook: /Users/kelvin/git_projects/market_helper/data/artifacts/portfolio_monitor/target_report.xlsx
Position report output: /Users/kelvin/git_projects/market_helper/outputs/reports/current_portfolio_report_preview.csv
Generated security reference: /Users/kelvin/git_projects/market_helper/configs/portfolio_monitor/security_reference.csv
Default live account: U2935967


In [2]:
from collections import defaultdict
import csv
import json
from typing import Dict, List, Tuple
from zipfile import ZipFile
import xml.etree.ElementTree as ET

from IPython.display import HTML, Markdown, display
import pandas as pd

from market_helper.portfolio.ibkr import (
    normalize_ibkr_latest_prices,
    normalize_ibkr_positions,
)
from market_helper.portfolio.security_reference import (
    build_price_lookup,
    build_security_reference_table,
    sync_security_reference_csv,
)
from market_helper.presentation.exporters.csv import export_position_report_csv
from market_helper.presentation.tables.portfolio_report import build_position_report_rows
from market_helper.reporting.risk_html import (
    _build_eq_country_breakdown,
    _build_fi_tenor_breakdown,
    _build_us_sector_breakdown,
    _funded_aum,
    build_risk_html_report,
    load_position_rows,
)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)

FUTURE_EXCHANGES = {"CBOT", "CFE", "CME", "COMEX", "ICE", "NYMEX"}

def infer_sec_type_from_row(row: Dict[str, str]) -> str:
    exchange = str(row.get("exchange") or "").upper()
    symbol = str(row.get("symbol") or "").upper()
    if symbol.endswith("_CASH_VALUE"):
        return "CASH"
    if exchange in FUTURE_EXCHANGES:
        return "FUT"
    return "STK"

def rebuild_position_report_from_existing_live_csv(
    existing_csv_path: Path,
    output_path: Path,
) -> Tuple[Path, object]:
    rows = list(csv.DictReader(existing_csv_path.open("r", encoding="utf-8", newline="")))
    if not rows:
        raise ValueError(f"No rows found in {existing_csv_path}")

    as_of = str(rows[0].get("as_of") or "") or None
    raw_positions: List[Dict[str, str]] = []
    raw_prices: List[Dict[str, str]] = []
    skipped_rows = 0

    for row in rows:
        con_id = str(row.get("con_id") or "").strip()
        sec_type = infer_sec_type_from_row(row)
        if not con_id and sec_type != "CASH":
            skipped_rows += 1
            continue
        raw_positions.append({
            "account": str(row.get("account") or ""),
            "con_id": con_id,
            "sec_type": sec_type,
            "symbol": str(row.get("symbol") or ""),
            "currency": str(row.get("currency") or ""),
            "exchange": str(row.get("exchange") or ""),
            "local_symbol": str(row.get("local_symbol") or ""),
            "position": str(row.get("quantity") or "0"),
            "avg_cost": str(row.get("avg_cost") or ""),
            "market_value": str(row.get("market_value") or ""),
        })
        if con_id and str(row.get("latest_price") or "").strip():
            raw_prices.append({
                "con_id": con_id,
                "marketPrice": str(row.get("latest_price") or ""),
            })

    reference_table = build_security_reference_table(reference_path=SECURITY_REFERENCE_OUTPUT)
    positions = normalize_ibkr_positions(raw_positions, reference_table, as_of=as_of)
    prices = normalize_ibkr_latest_prices(raw_prices, reference_table, as_of=as_of)
    report_rows = build_position_report_rows(
        positions,
        build_price_lookup(prices),
        reference_table.to_security_lookup(),
    )
    written_path = export_position_report_csv(report_rows, output_path)
    print(
        f"Rebuilt {len(report_rows)} normalized report rows from existing CSV "
        f"({skipped_rows} skipped rows without usable contract ids)."
    )
    return written_path, reference_table

def breakdown_df(rows: List[object]) -> pd.DataFrame:
    return pd.DataFrame([row.__dict__ for row in rows])

def preview_target_workbook_sheet(workbook_path: Path, sheet_name: str, max_rows: int = 10, max_cols: int = 14) -> pd.DataFrame:
    ns = {
        "a": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
        "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
        "pr": "http://schemas.openxmlformats.org/package/2006/relationships",
    }
    with ZipFile(workbook_path) as archive:
        workbook = ET.fromstring(archive.read("xl/workbook.xml"))
        rels = ET.fromstring(archive.read("xl/_rels/workbook.xml.rels"))
        relmap = {
            rel.attrib["Id"]: rel.attrib["Target"]
            for rel in rels.findall("pr:Relationship", ns)
        }
        shared_strings: List[str] = []
        if "xl/sharedStrings.xml" in archive.namelist():
            shared_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
            shared_strings = [
                "".join(text.text or "" for text in item.iterfind(".//a:t", ns))
                for item in shared_root.findall("a:si", ns)
            ]
        sheets = workbook.find("a:sheets", ns)
        matching = [sheet for sheet in sheets if sheet.attrib.get("name") == sheet_name]
        if not matching:
            raise KeyError(f"Workbook sheet not found: {sheet_name}")
        rid = matching[0].attrib["{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"]
        target = relmap[rid]
        if not target.startswith("xl/"):
            target = f"xl/{target}"
        root = ET.fromstring(archive.read(target))
        materialized: List[List[object]] = []
        for row in root.findall(".//a:sheetData/a:row", ns)[:max_rows]:
            values: List[object] = []
            for cell in row.findall("a:c", ns)[:max_cols]:
                value = cell.find("a:v", ns)
                if value is None:
                    values.append(None)
                    continue
                if cell.attrib.get("t") == "s":
                    idx = int(value.text)
                    values.append(shared_strings[idx] if idx < len(shared_strings) else value.text)
                else:
                    values.append(value.text)
            materialized.append(values)
        return pd.DataFrame(materialized)


In [3]:
synced_reference_path = sync_security_reference_csv(reference_path=SECURITY_REFERENCE_OUTPUT)
print(f"Synced generated security reference to: {synced_reference_path}")

if SOURCE_MODE == "live":
    from market_helper.workflows.generate_report import generate_live_ibkr_position_report

    position_report_path = generate_live_ibkr_position_report(
        output_path=POSITION_REPORT_OUTPUT,
        host=IB_HOST,
        port=IB_PORT,
        client_id=IB_CLIENT_ID,
        account_id=IB_ACCOUNT or None,
        timeout=4.0,
    )
    reference_table = build_security_reference_table(reference_path=synced_reference_path)
    acquisition_note = "Generated directly from live TWS / IB Gateway."
else:
    if not EXISTING_LIVE_CSV.exists():
        raise FileNotFoundError(
            f"Fallback CSV not found at {EXISTING_LIVE_CSV}. Either switch SOURCE_MODE to live or provide a local CSV."
        )
    position_report_path, reference_table = rebuild_position_report_from_existing_live_csv(
        existing_csv_path=EXISTING_LIVE_CSV,
        output_path=POSITION_REPORT_OUTPUT,
    )
    acquisition_note = "Rebuilt offline from an existing live IBKR report CSV through the current normalization stack."

position_report_df = pd.read_csv(position_report_path)
risk_input_rows = load_position_rows(position_report_path, security_reference_table=reference_table)
risk_input_df = pd.DataFrame([row.__dict__ for row in risk_input_rows])

print(acquisition_note)
print(f"Position report path: {position_report_path}")
print(f"Position rows: {len(position_report_df)}")
mapped_rows = int((risk_input_df["mapping_status"] == "mapped").sum())
print(f"Mapped rows: {mapped_rows} / {len(risk_input_df)}")

preview_columns = [
    "account",
    "internal_id",
    "symbol",
    "display_ticker",
    "display_name",
    "asset_class",
    "quantity",
    "market_value",
    "signed_exposure_usd",
    "eq_country",
    "eq_sector",
    "fi_tenor",
    "mapping_status",
]
display(
    risk_input_df.loc[:, preview_columns]
    .sort_values("market_value", ascending=False, na_position="last")
    .reset_index(drop=True)
)


Synced generated security reference to: /Users/kelvin/git_projects/market_helper/configs/portfolio_monitor/security_reference.csv
Rebuilt 26 normalized report rows from existing CSV (0 skipped rows without usable contract ids).
Rebuilt offline from an existing live IBKR report CSV through the current normalization stack.
Position report path: /Users/kelvin/git_projects/market_helper/outputs/reports/current_portfolio_report_preview.csv
Position rows: 26
Mapped rows: 25 / 26


,account,internal_id,symbol,display_ticker,display_name,asset_class,quantity,market_value,signed_exposure_usd,eq_country,eq_sector,fi_tenor,mapping_status
0,U2935967,FUT:ZT:CBOT,ZT,ZT,2Y TF,FI,6.0,1242000.00,1242000.00,,,1-3Y,mapped
1,U2935967,FUT:ZF:CBOT,ZF,ZF,5Y TF,FI,4.0,431250.00,431250.00,,,3-5Y,mapped
2,U2935967,FUT:AUD:CME,AUD,AUD,AUD FX Future,FX,6.0,415067.98,415067.98,,,,mapped
3,U2935967,FUT:EUR:CME,EUR,EUR,EUR FX Future,FX,1.0,144809.99,144809.99,,,,mapped
4,U2935967,FUT:ZN:CBOT,ZN,ZN,10Y TF,FI,1.0,110625.00,110625.00,,,7-10Y,mapped
5,U2935967,STK:VOO:SMART,VOO,VOO,US,EQ,170.0,102574.32,102574.32,US,,,mapped
6,U2935967,STK:ACWD:LSEETF,ACWD,ACWD,ACWI,EQ,300.0,84702.06,84702.06,ACWI,,,mapped
7,U2935967,FUT:GBP:CME,GBP,GBP,GBP FX Future,FX,1.0,83453.56,83453.56,,,,mapped
8,U2935967,STK:GLD:SMART,GLD,GLD,Gold,CM,185.0,75533.66,75533.66,,,,mapped
9,U2935967,STK:IEUR:SMART,IEUR,IEUR,EU,EQ,1000.0,69133.68,69133.68,EU,,,mapped


In [4]:
funded_aum = _funded_aum(risk_input_rows)
asset_class_summary = (
    risk_input_df.groupby("asset_class", as_index=False)
    .agg(
        positions=("internal_id", "count"),
        net_exposure_usd=("signed_exposure_usd", "sum"),
        gross_exposure_usd=("gross_exposure_usd", "sum"),
        market_value_usd=("market_value", "sum"),
    )
    .sort_values("gross_exposure_usd", ascending=False)
    .reset_index(drop=True)
)
asset_class_summary["gross_pct_of_aum"] = asset_class_summary["gross_exposure_usd"] / funded_aum

top_positions = (
    risk_input_df.loc[:, [
        "display_ticker",
        "display_name",
        "asset_class",
        "quantity",
        "market_value",
        "signed_exposure_usd",
        "eq_country",
        "eq_sector",
        "fi_tenor",
    ]]
    .sort_values("market_value", ascending=False, key=lambda col: col.abs())
    .head(15)
    .reset_index(drop=True)
)

zero_loadings = {row.internal_id: 0.0 for row in risk_input_rows}
country_breakdown = breakdown_df(_build_eq_country_breakdown(risk_input_rows, zero_loadings))
sector_breakdown = breakdown_df(_build_us_sector_breakdown(risk_input_rows, zero_loadings))
fi_tenor_breakdown = breakdown_df(_build_fi_tenor_breakdown(risk_input_rows, zero_loadings))

portfolio_overview = pd.DataFrame(
    [
        {"metric": "Funded AUM", "value": funded_aum},
        {"metric": "Net exposure", "value": float(risk_input_df["signed_exposure_usd"].sum())},
        {"metric": "Gross exposure", "value": float(risk_input_df["gross_exposure_usd"].sum())},
        {"metric": "Mapped rows", "value": int((risk_input_df["mapping_status"] == "mapped").sum())},
        {"metric": "Total rows", "value": int(len(risk_input_df))},
    ]
)

display(Markdown("## Current Portfolio Report Pieces"))
display(portfolio_overview)

display(Markdown("### Top Positions"))
display(top_positions)

display(Markdown("### Asset Class Summary"))
display(asset_class_summary)

display(Markdown("### EQ Country Breakdown"))
display(country_breakdown)

display(Markdown("### US Sector Breakdown"))
display(sector_breakdown if not sector_breakdown.empty else pd.DataFrame([{"bucket": "No data", "parent": "US_EQ"}]))

display(Markdown("### FI Tenor Breakdown"))
display(fi_tenor_breakdown)


## Current Portfolio Report Pieces

,metric,value
0,Funded AUM,741109.24
1,Net exposure,3160103.87
2,Gross exposure,3168315.77
3,Mapped rows,25.00
4,Total rows,26.00


### Top Positions

,display_ticker,display_name,asset_class,quantity,market_value,signed_exposure_usd,eq_country,eq_sector,fi_tenor
0,ZT,2Y TF,FI,6.0,1242000.00,1242000.00,,,1-3Y
1,ZF,5Y TF,FI,4.0,431250.00,431250.00,,,3-5Y
2,AUD,AUD FX Future,FX,6.0,415067.98,415067.98,,,
3,EUR,EUR FX Future,FX,1.0,144809.99,144809.99,,,
4,ZN,10Y TF,FI,1.0,110625.00,110625.00,,,7-10Y
5,VOO,US,EQ,170.0,102574.32,102574.32,US,,
6,ACWD,ACWI,EQ,300.0,84702.06,84702.06,ACWI,,
7,GBP,GBP FX Future,FX,1.0,83453.56,83453.56,,,
8,GLD,Gold,CM,185.0,75533.66,75533.66,,,
9,IEUR,EU,EQ,1000.0,69133.68,69133.68,EU,,


### Asset Class Summary

...

,bucket,bucket_label,parent,exposure_usd,gross_exposure_usd,dollar_weight,risk_contribution_estimated
0,US,,EQ,341689.2156,349901.1156,0.472132,0.0
1,EU,,EQ,89615.1980,89615.1980,0.120920,0.0
2,CN,,EQ,56103.7534,56103.7534,0.075702,0.0
3,KR,,EQ,32076.4167,32076.4167,0.043282,0.0
4,OTHERS,,EQ,18853.7807,18853.7807,0.025440,0.0
5,TW,,EQ,10851.7758,10851.7758,0.014643,0.0
6,JP,,EQ,10154.6116,10154.6116,0.013702,0.0
7,IN,,EQ,5849.3982,5849.3982,0.007893,0.0


### US Sector Breakdown

,bucket,bucket_label,parent,exposure_usd,gross_exposure_usd,dollar_weight,risk_contribution_estimated
0,Technology,,US_EQ,50807.4407,53353.1297,0.071991,0.0
1,Financials,,US_EQ,22945.2958,24094.9618,0.032512,0.0
2,Health Care,,US_EQ,19667.3964,20652.8244,0.027867,0.0
3,Semiconductor,,US_EQ,20246.7100,20246.7100,0.027319,0.0
4,Consumer Discretionary,,US_EQ,16389.4970,17210.6870,0.023223,0.0
5,Communication Services,,US_EQ,14750.5473,15489.6183,0.020901,0.0
6,Industrials,,US_EQ,13111.5976,13768.5496,0.018578,0.0
7,Consumer Staples,,US_EQ,9833.6982,10326.4122,0.013934,0.0
8,Energy,,US_EQ,6555.7988,6884.2748,0.009289,0.0
9,Utilities,,US_EQ,4916.8491,5163.2061,0.006967,0.0


### FI Tenor Breakdown

,bucket,bucket_label,parent,exposure_usd,gross_exposure_usd,dollar_weight,risk_contribution_estimated
0,1-3Y,Front end,FI,1242000.0,1242000.0,1.675866,0.0
1,3-5Y,Short belly,FI,431250.0,431250.0,0.581898,0.0
2,7-10Y,Long belly,FI,154017.0,154017.0,0.207820,0.0


In [5]:
risk_html_path = None
if TRY_BUILD_RISK_HTML:
    try:
        risk_html_path = build_risk_html_report(
            positions_csv_path=position_report_path,
            output_path=RISK_HTML_OUTPUT,
            returns_path=Path(RETURNS_JSON) if RETURNS_JSON else None,
            proxy_path=Path(PROXY_JSON) if PROXY_JSON else None,
            regime_path=Path(REGIME_JSON) if REGIME_JSON else None,
            security_reference_path=SECURITY_REFERENCE_OUTPUT,
        )
        print(f"Generated HTML risk report: {risk_html_path}")
    except Exception as exc:
        print("Risk HTML generation is optional in this notebook and failed cleanly:")
        print(type(exc).__name__, exc)
else:
    print("TRY_BUILD_RISK_HTML is False. Set it to True if you want to attempt the HTML risk report build.")


TRY_BUILD_RISK_HTML is False. Set it to True if you want to attempt the HTML risk report build.


In [6]:
position_sheet_preview = preview_target_workbook_sheet(TARGET_WORKBOOK_PATH, "Position")
risk_sheet_preview = preview_target_workbook_sheet(TARGET_WORKBOOK_PATH, "Risk")

gap_review = pd.DataFrame(
    [
        {
            "layer": "Universe-driven security semantics",
            "status": "done",
            "what_you_can_see_now": "Stable IDs, generated security reference, and semantic fields come from security_universe.csv.",
            "main_gap": "Add local override workflow and broader rule coverage.",
        },
        {
            "layer": "Normalized current position report",
            "status": "done",
            "what_you_can_see_now": "Current notebook can build a normalized report from live TWS or from an existing live CSV.",
            "main_gap": "Improve live ergonomics and fixture coverage.",
        },
        {
            "layer": "Exposure bucket rollups",
            "status": "mostly done",
            "what_you_can_see_now": "Asset class, EQ country, US sector, and FI tenor sections are available today.",
            "main_gap": "Move from preview tables to final workbook layout.",
        },
        {
            "layer": "Risk HTML",
            "status": "partially done",
            "what_you_can_see_now": "Optional HTML report can be generated when returns/proxy data are available.",
            "main_gap": "Cache returns/proxies and add deeper attribution.",
        },
        {
            "layer": "Target workbook Position sheet parity",
            "status": "not done",
            "what_you_can_see_now": "We have the core data but not the final workbook sections/layout.",
            "main_gap": "Workbook renderer and target-style subtotal sections.",
        },
        {
            "layer": "Target workbook Risk sheet parity",
            "status": "not done",
            "what_you_can_see_now": "We have partial risk analytics and regime integration, but not target parity.",
            "main_gap": "Covariance attribution, scenario panels, and workbook formatting.",
        },
    ]
)

display(Markdown("## Target Workbook Preview"))
display(Markdown("### Position sheet (top rows)"))
display(position_sheet_preview)

display(Markdown("### Risk sheet (top rows)"))
display(risk_sheet_preview)

display(Markdown("## Gap Review"))
display(gap_review)

display(Markdown("**Practical read:** useful internal portfolio report is close; target workbook parity is still a later-stage formatting + attribution project."))


## Target Workbook Preview

### Position sheet (top rows)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,Category,Ticker,Name,Qty,Multiplier,Price,Exposure(USD),Type,duration (FUT_EQV_MOD_DUR_BASED_ON_CTD),FI Exposure (10Y eqv),Dollar%,FX,Expected Vol,Vol Contribution
1,Summary,%funded,0.936717487139845,DMEQ,SPY,US,100,1,656.82,65682,ETF,1,None,0.0852718803870645
2,$AUM,770265.645625,DMEQ,VOO,US,155,1,605.66,93877.3,ETF,1,None,0.121876524720023,1
3,S$ AUM,987634.611446535,DMEQ,LON:SPYL,US,4000,1,16.26,65040,ETF,1,None,0.0844384016987101,1
4,DMEQ,SOXX,US,60,1,345.25,20715,ETF,1,None,0.0268933193602211,1,0.18,0.00484079748483979
5,Dollar Allocation,$,Dollar%,vol contr,Wt in ACWI,DMEQ,TSLA,TSLA,15,1,385.95,5789.25,ETF,1
6,DMEQ,448015.13,0.581637169650059,0.0765320481769214,0.89,DMEQ,IEUR,EU,1000,1,69.71,69710,ETF,1
7,EMEQ,118197.67,0.153450527972195,0.0312768070818633,0.11,DMEQ,VEA,DM,800,1,63.97,51176,ETF,1
8,EQ,566212.8,0.735087697622254,0.107808855258785,None,None,None,None,None,None,None,None,None,None
9,FI,709617.225383506,0.921262981692136,0.0686712958061117,None,,None,EMEQ,LON:XMME,EM,200,1,79.51,15902


### Risk sheet (top rows)

,0,1,2,3,4,5,6,7,8,9,10
0,31,92,365,1826,None,None,None,None,None,None,None
1,Latest,1M_AVG,3M_AVG,1Y_AVG,5Y_AVG,EST,Tail,daily move,None,None,None
2,EQ,INDEXCBOE:VIX,26.95,23.655909090909,19.3091803278688,19.2438399999999,19.1748363926576,21.3723235624027,40,%,1.33838681717081
3,FI,INDEXNYSEGIS:MOVE,103.01,80.5254545454545,68.5444262295082,82.2746747967479,99.8146451612902,74.2938158845575,130,bps,4.65245921843176
4,GOLD,INDEXCBOE:GVZ,38.65,None,None,None,None,None,None,None,None
5,OIL,INDEXCBOE:OVX,90.16,None,None,None,None,None,None,None,None
6,None,None,None,None,None,None,None,None,None,None,None
7,Latest,1M_AVG,3M_AVG,1Y_AVG,5Y_AVG,EST,Tail,None,None,None,None
8,None,EQ Risk Attr,0.198106134509197,0.173891677487976,0.14193940910186,0.14145910039011,0.140951863361621,0.157105321203244,0.294035079048901,None,None
9,None,FI Risk Attr,0.0759194397952856,0.059348096285328,0.0505179539892587,0.0606372897768023,0.0735644300942662,0.0547553138744752,0.0958113500959822,None,None


## Gap Review

,layer,status,what_you_can_see_now,main_gap
0,Universe-driven security semantics,done,"Stable IDs, generated security reference, and ...",Add local override workflow and broader rule c...
1,Normalized current position report,done,Current notebook can build a normalized report...,Improve live ergonomics and fixture coverage.
2,Exposure bucket rollups,mostly done,"Asset class, EQ country, US sector, and FI ten...",Move from preview tables to final workbook lay...
3,Risk HTML,partially done,Optional HTML report can be generated when ret...,Cache returns/proxies and add deeper attribution.
4,Target workbook Position sheet parity,not done,We have the core data but not the final workbo...,Workbook renderer and target-style subtotal se...
5,Target workbook Risk sheet parity,not done,We have partial risk analytics and regime inte...,"Covariance attribution, scenario panels, and w..."


**Practical read:** useful internal portfolio report is close; target workbook parity is still a later-stage formatting + attribution project.